[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/klein_quadric/klein_quadric.ipynb)

# Four Lines and the Klein Quadric

This example finds the lines that meet four given lines in space. Three lines in general position have a whole family of lines meeting all three, sweeping out a hyperboloid; a fourth line crosses that hyperboloid in two points, and through each passes one line meeting all four. When the fourth line misses the hyperboloid, the two lines are still there, as a complex-conjugate pair.

A line in space is modelled as a bivector of projective geometric algebra, but most bivectors are not lines: a bivector is a line when its wedge with itself vanishes, and two lines meet when their wedge vanishes. Both are one form, `Bivector ^ Bivector`, a quadratic form on the six-dimensional space of bivectors whose zeros are the lines: the Klein quadric. Meeting four lines is four linear conditions, which leave a pencil of bivectors; along the pencil the Klein form is a quadratic, and its two zeros are the two transversals. The algebra runs on complex coefficients, so that both are there in every case.

In Plücker coordinates a line reads as six numbers $p_{ij}$ on the Klein quadric $p_{01}p_{23} - p_{02}p_{13} + p_{03}p_{12} = 0$ in five-dimensional projective space, and in the Schubert calculus of the Grassmannian of lines the count of two reads as the intersection number $\sigma_1^4 = 2$ of four Schubert cycles.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure, show_animation
from examples.math.klein_quadric import render
from examples.math.klein_quadric.core import spread
from examples.math.klein_quadric.scenarios import paced

In [ ]:
# Complex coefficients, so that the two transversals are there in every case, a conjugate pair where they are not real.
mv = NumpyContext(ga, np.complex128).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
# Every line here is a meet of two planes, as the lines through a point that meet two lines are: an axis.
Bivector = ga.gatype.bivector()                                                # Bivector+
Point = ga.gatype.antivector()                                                 # Antivector-
# A point joined with a line and met with others gives a plane through points: oriented within itself.
Quadric = ga.gatype((ga.gatype.antitrivector(), Point))
# Zero on one bivector twice exactly when it is a line; zero on two lines exactly when they meet.
klein = Bivector ^ Bivector                                                    # [] Pseudoscalar+ <- (Bivector+, Bivector+)
# The origin, the point dual to the plane at infinity; the plane at infinity as the join of the three directions,
# whose meet with an axis is the axis's point at infinity.
origin = mv.w.dual()                                                           # [] Point-
horizon = mv.direction([1, 0, 0]) & mv.direction([0, 1, 0]) & mv.direction([0, 0, 1])   # [] Plane-
# Both roots of a quadratic, one per sign.
signs = np.array([1, -1])

## 1. The Klein form

The Klein form takes two bivectors to their wedge, a pseudoscalar. A meet of two planes is a line and the form vanishes on it twice; a sum of two skew lines is not a line, and the form does not. Two lines that cross pair to zero, and two that pass each other do not.

In [ ]:
# Two lines along z, through the origin and through (1, 0, 0), each the meet of two planes; and a line
# along y through (1, 0, 1), passing the first.
along_z = mv.xy                                                                # [] Bivector+
beside = (mv.x - mv.w) ^ mv.y                                                  # [] Bivector+
above = (mv.x - mv.w) ^ (mv.z - mv.w)                                          # [] Bivector+

## 2. Four lines, two transversals

Meeting a line is the linear condition `line ^ bivector == 0`, so the bivectors that meet four lines form a pencil, and the Klein form along the pencil is a quadratic whose two zeros are the transversals. Three lines are fixed through the waist of the hyperboloid $x^2 + y^2 = 1 + z^2$, and a steep fourth line is placed once outside the waist and once through it. Outside the waist both transversals are real; through it they are a complex-conjugate pair, still two lines that meet all four.

In [ ]:
# Where the three lines cross the hyperboloid's waist, by angle about z; where the fourth line crosses the plane
# z = 0 at rest and the direction it runs in; and its offsets along x: outside the waist, and through it.
waist_angles = np.array([0.3, 2.3, 4.4])
fourth_crossing, fourth_direction = [0.0, 0.4, 0.0], [0.3, -0.2, 1.0]
offsets = np.array([1.4, 0.0])

In [ ]:
def roots(b: Scalar, c: Scalar) -> Scalar:
    """The two roots of the quadratic `t * t + 2 * b * t + c` along the combinations of two ends,
    as the weight of the first end against unit weight on the second, one root per sign on the last
    axis: `first * (sign * root - b) + second`, with `root * root == b * b - c`. A quadratic whose
    coefficients share a factor, such as the handedness of a signed volume, is divided by its first
    coefficient to read as this one."""
    root = (b * b - c).square_root()                                           # [...] Scalar+
    return root[..., None] * signs - b[..., None]                              # [..., 2] Scalar+


def transversals_of(lines: Bivector) -> Bivector:
    """The two lines that meet all four lines on the last axis, at unit norm, on the last axis."""
    # Each line, weighted by the Klein form between it and a bivector: zero exactly on the bivectors
    # that meet all four.
    misses = (lines * klein(lines).dual()).sum(axis=-1)                        # [...] Bivector- <- Bivector+
    # Those bivectors form a pencil, spanned by the right singular vectors of zero singular value.
    _, _, right = misses.svd()
    pencil = right[..., -2:]                                                   # [..., 2] Bivector+
    # Along the pencil the Klein form is a quadratic; its two zeros are the lines of the pencil.
    first, second = pencil[..., 0], pencil[..., 1]                             # [...] Bivector+, Bivector+
    # The Klein form of two axes is a signed volume; its ratios are ordinary numbers.
    leading = klein(first, first).dual()                                       # [...] Scalar-
    weights = roots(klein(first, second).dual() / leading, klein(second, second).dual() / leading)   # [..., 2] Scalar+
    return (first[..., None] * weights + second[..., None]).normalized()       # [..., 2] Bivector+


def steep(offsets: np.ndarray) -> Bivector:
    """The fourth line, moved along x by each offset: the meet of the two planes through it along x and along y."""
    shifts = (mv.xw * (offsets / 2)).exp()                                     # [offsets] Motor+
    anchors = shifts >> mv.point(fourth_crossing)                              # [offsets] Point-
    # The fourth line as the join of its anchor and its direction, and again as a meet of planes.
    line = anchors & mv.direction(fourth_direction)                            # [offsets] Line-
    return (line & mv.direction([1, 0, 0])) ^ (line & mv.direction([0, 1, 0]))   # [offsets] Bivector+


# The line through (1, 0, 0) leaning along (0, 1, 1), the meet of the planes x = 1 and y = z, turned about z to each
# point of the waist: lines of the hyperboloid, all leaning the same way.
leaning = (mv.x - mv.w) ^ (mv.y - mv.z)                                        # [] Bivector+
waist_turns = (mv.xy * (-waist_angles / 2)).exp()                              # [3] Rotor+
three = waist_turns >> leaning                                                 # [3] Bivector+
fourth = steep(offsets)                                                        # [offsets] Bivector+
lines = stack([three[0].broadcast_to(fourth.shape), three[1].broadcast_to(fourth.shape),
               three[2].broadcast_to(fourth.shape), fourth], axis=-1)          # [offsets, 4] Bivector+
transversals = transversals_of(lines)                                          # [offsets, 2] Bivector+

## 3. The same count, seen in space

The lines that meet three lines sweep out a quadric. As a map from points to planes it pairs to zero with its points: the plane joining a point to the first line meets the third line in a point, and the plane joining that point to the second line holds the first point exactly when a line through it meets all three. The fourth line crosses that quadric at two points, by the same quadratic as before, now along the fourth line, and through each crossing runs one of the transversals. In the animation the fourth line, orange, swings into the hyperboloid's waist and out again. While it passes outside the waist it crosses the surface at two points, and through each crossing runs a red transversal. As it nears the waist the two crossings slide together, and the two red lines with them; where the orange line touches the surface they coincide, and past that point they leave the real picture together, as a complex-conjugate pair. On the way out they return the same way. The swing slows where the crossings merge, so that the red pair is seen to close and vanish.

In [ ]:
# The lines drawn of each family; the frames of the swing and how far it reaches either way; the points the swing
# is sampled at to pace it, and the share of the line's own motion in the pace; the frame shown as a still; and the
# duration of a frame in milliseconds.
n_rulings = 12
n_frames = 168
swing_amplitude = 1.5
n_path, line_share = 960, 0.6
still_frame = n_frames // 12
frame_duration_ms = 40

In [ ]:
def crossings_of(surface: Quadric, line: Bivector) -> Point:
    """The two points where a line crosses a quadric, at unit weight, on the last axis."""
    # The line's point nearest the origin and its point at infinity, its direction, span it.
    nearest = (origin | line) ^ line                                           # [...] Point-
    direction = line ^ horizon                                                 # [...] Point-
    # The quadric's map need not be symmetric: the cross term is the mean of both orders.
    cross = ((surface(nearest) & direction) + (surface(direction) & nearest)) / 2   # [...] Scalar-
    leading = surface(nearest) & nearest                                       # [...] Scalar-
    weights = roots(cross / leading, (surface(direction) & direction) / leading)   # [..., 2] Scalar+
    crossings = nearest[..., None] * weights + direction[..., None]            # [..., 2] Point-
    return crossings.normalized()                                              # [..., 2] Point-


surface = ((Point & three[0]) ^ three[2]) & three[1]                           # [] Plane- <- Point-
# The lines through points spread over the first line that meet the other two, each the meet of the planes joining
# its point to those lines; and the lines that meet three of those, the family of the three.
starts = spread(three[0], n_rulings)                                           # [rulings] Point-
sweep = (starts & three[1]) ^ (starts & three[2])                              # [rulings] Bivector+
family_starts = spread(sweep[0], n_rulings)                                    # [rulings] Point-
family = (family_starts & sweep[n_rulings // 3]) ^ (family_starts & sweep[2 * n_rulings // 3])   # [rulings] Bivector+
rulings = stack([sweep, family])                                               # [families, rulings] Bivector+
# The swing sampled closely along a closed path, paced by the distance between the crossings, whose square turns
# negative where they are a complex pair.
path = swing_amplitude * np.cos(np.linspace(0.0, 2 * np.pi, n_path + 1))
along_path = crossings_of(surface, steep(path))                                # [path + 1, 2] Point-
separations = (along_path[..., 0] & along_path[..., 1]).scalar_norm_squared()  # [path + 1] Scalar+
frame_offsets = paced(path, separations, n_frames, line_share)


def swing(frame_offsets: np.ndarray) -> Iterator[tuple[Bivector, Bivector, Bivector, Point]]:
    """Per frame, the four lines, both families of lines on the hyperboloid, the transversals and the crossings."""
    for offset in frame_offsets:
        fourth_line = steep(offset)                                            # [] Bivector+
        frame_lines = stack([three[0], three[1], three[2], fourth_line])       # [4] Bivector+
        yield frame_lines, rulings, transversals_of(frame_lines), crossings_of(surface, fourth_line)


save_figure(render.draw(*next(swing(frame_offsets[still_frame:]))), "klein_transversals")
show_animation(render.animate(swing(frame_offsets), n_frames), "klein_swing", frame_duration_ms, n_frames)

In [ ]:
# checks
# The Klein form vanishes on a line and on two lines that meet, and not on a sum of skew lines or on two skew lines.
np.testing.assert_allclose(klein(along_z, along_z).dual().to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose(klein(along_z, beside).dual().to_array(), 0.0, atol=1e-12)
assert np.abs(klein(along_z + above, along_z + above).dual().to_array()) > 0.5
assert np.abs(klein(along_z, above).dual().to_array()) > 0.5
# The transversals are lines and meet all four; outside the waist both are real, through it both are complex.
np.testing.assert_allclose((transversals ^ transversals).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((lines[..., :, None] ^ transversals[..., None, :]).kernel, 0.0, atol=1e-12)
real = np.abs(transversals.kernel.imag).max(axis=-1) <= 1e-6 * np.abs(transversals.kernel).max(axis=-1)   # [offsets, 2]
assert real[0].all() and not real[1].any()
# The crossings lie on the quadric and on a transversal each; both families lie on the quadric.
crossings = crossings_of(surface, fourth)                                      # [offsets, 2] Point-
np.testing.assert_allclose((surface(along_path) & along_path).kernel, 0.0, atol=1e-10)
off = np.abs((crossings[..., :, None] & transversals[..., None, :]).kernel).max(axis=-1).min(axis=-1)   # [offsets, 2]
np.testing.assert_allclose(off, 0.0, atol=1e-10)
ends = spread(rulings, 3)                                                      # [families, rulings, 3] Point-
np.testing.assert_allclose((surface(ends) & ends).kernel, 0.0, atol=1e-10)